# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kbhutto256/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

For the March 15 decision point, I use five public-safe Google Search Console features calculated only from **March 1–15, 2026**. The modeling grain is one row per `client_hash_id × content_hash_id`. The feature vector is deliberately small and readable: impressions, clicks, average position, active days, and CTR.

This notebook uses a frozen public-safe feature specification from the executed March data contract rather than publishing row-level warehouse data. The actual warehouse rebuild remains in `w03_data_contract.ipynb` and later modeling notebooks.

In [1]:
import pandas as pd

feature_spec = pd.DataFrame([
    ["first_half_impressions", "numeric", "sum of GSC impressions, Mar 1–15", 0, True],
    ["first_half_clicks", "numeric", "sum of GSC clicks, Mar 1–15", 0, True],
    ["first_half_avg_position", "numeric", "impression-weighted average position, Mar 1–15", None, True],
    ["first_half_active_days", "numeric", "count of days with available GSC data, Mar 1–15", 0, True],
    ["first_half_ctr_pct", "numeric", "100 × clicks / impressions, Mar 1–15", 0.0, True],
], columns=["feature", "type", "meaning", "fill_if_missing", "available_by_decision_time"])

FEATURE_COLS = feature_spec["feature"].tolist()
print("Feature count:", len(FEATURE_COLS))
display(feature_spec)


Feature count: 5


,feature,type,meaning,fill_if_missing,available_by_decision_time
0,first_half_impressions,numeric,"sum of GSC impressions, Mar 1–15",0.0,True
1,first_half_clicks,numeric,"sum of GSC clicks, Mar 1–15",0.0,True
2,first_half_avg_position,numeric,"impression-weighted average position, Mar 1–15",NaN,True
3,first_half_active_days,numeric,"count of days with available GSC data, Mar 1–15",0.0,True
4,first_half_ctr_pct,numeric,"100 × clicks / impressions, Mar 1–15",0.0,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

All five predictors are numeric and are available by the **March 15, 2026** decision point. I do not use client or content identifiers as predictors.

Missing handling is conservative and follows the data contract: rows are included only where `gsc_data_available IS TRUE`. Count-like fields can use zero after that availability filter when an observed page-day has no clicks/impressions; average position is left missing when it is not meaningfully defined and is handled later by the model pipeline's imputer. CTR is computed from the same pre-decision window only.

There are no categorical predictors in this version, so no one-hot encoding is required.

In [2]:
notes = feature_spec[["feature", "type", "fill_if_missing", "available_by_decision_time"]].copy()
assert notes["available_by_decision_time"].all(), "A feature is not available by the decision point."
assert (notes["type"] == "numeric").all(), "Unexpected categorical feature found."
print("Availability check passed: every predictor is pre-decision and numeric.")
display(notes)


Availability check passed: every predictor is pre-decision and numeric.


,feature,type,fill_if_missing,available_by_decision_time
0,first_half_impressions,numeric,0.0,True
1,first_half_clicks,numeric,0.0,True
2,first_half_avg_position,numeric,NaN,True
3,first_half_active_days,numeric,0.0,True
4,first_half_ctr_pct,numeric,0.0,True


## 3. The leakage hunt

The label is a forward visibility-decline proxy: `decline_proxy = 1` when March 16–31 impressions are below 80% of March 1–15 impressions. Therefore any second-half metric, future-derived trend field, or label-derived field is leakage.

I also treat raw client/content identifiers as context only. They are useful for grouping and leakage-safe validation, but they are not meaningful predictive features. The audit below explicitly checks the candidate feature list against forbidden future/label fields and identifier fields.

In [3]:
candidate_features = FEATURE_COLS.copy()

forbidden_future_or_label = {
    "decline_proxy",
    "second_half_impressions",
    "second_half_clicks",
    "second_half_avg_position",
    "second_half_active_days",
    "second_half_ctr_pct",
    "future_trend_pct",
    "trend_direction",
    "trend_pct",
}
identifier_fields = {"client_hash_id", "content_hash_id", "report_date"}

leaked = sorted(set(candidate_features) & forbidden_future_or_label)
ids_used = sorted(set(candidate_features) & identifier_fields)

print("Leakage fields found:", leaked)
print("Identifier predictors found:", ids_used)
assert not leaked, f"Leakage detected: {leaked}"
assert not ids_used, f"Identifier used as predictor: {ids_used}"
print("Leakage/privacy check passed for the committed feature list.")


Leakage fields found: []
Identifier predictors found: []
Leakage/privacy check passed for the committed feature list.


## 4. What I excluded and why

- **Second-half metrics (March 16–31):** they occur after the decision point and would leak outcome-window information.
- **`decline_proxy`:** this is the target, never a predictor.
- **`future_trend_pct`, `trend_direction`, `trend_pct`:** future/label-derived signals would make evaluation unrealistically easy.
- **`client_hash_id`, `content_hash_id`:** retained only for grouping/context; IDs are not substantive page features.
- **`report_date`:** used to construct windows, not as a model feature.
- **Client names, domains, URLs, page titles, and private queries:** not published or modeled in this public artifact.

The resulting feature set supports **directional decision-support** only. It does not establish why search visibility changes and does not predict Google's algorithm.

In [4]:
excluded = pd.DataFrame([
    ["second_half_*", "future outcome-window information"],
    ["decline_proxy", "target / label"],
    ["future_trend_pct", "future-derived leakage"],
    ["trend_direction", "label-derived leakage"],
    ["trend_pct", "label-derived leakage"],
    ["client_hash_id", "grouping/context ID, not a predictor"],
    ["content_hash_id", "context ID, not a predictor"],
    ["report_date", "window construction only"],
    ["client names / domains / URLs / private queries", "public-safety exclusion"],
], columns=["excluded_field", "reason"])

print("Excluded field groups:", len(excluded))
display(excluded)


Excluded field groups: 9


,excluded_field,reason
0,second_half_*,future outcome-window information
1,decline_proxy,target / label
2,future_trend_pct,future-derived leakage
3,trend_direction,label-derived leakage
4,trend_pct,label-derived leakage
5,client_hash_id,"grouping/context ID, not a predictor"
6,content_hash_id,"context ID, not a predictor"
7,report_date,window construction only
8,client names / domains / URLs / private queries,public-safety exclusion


## Self-check

Before submission:

- [x] Every section above is filled — markdown thinking AND code that backs it.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, or private queries are exposed.
- [x] Claims use careful words: observed, measured, directional, decision-support.
- [ ] Commit this executed notebook to `work/notebooks/w03_feature_leakage_check.ipynb` in the repo.

**Important:** This notebook audits the frozen public-safe feature specification. The warehouse-level feature-frame rebuild and observed March counts remain traceable to the executed data-contract/model notebooks; no row-level private warehouse data is embedded here.